# Model Training

Objective:
Build a machine learning pipeline capable of predicting SLA breaches at incident creation time.

The modeling phase includes:

- Preprocessing Pipeline
- Missing Value Handling
- Encoding
- Class Imbalance Handling
- Model Training
- Cross Validation
- Hyperparameter Tuning
- Model Selection

# Import Libraries

In [38]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder,TargetEncoder,FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,roc_auc_score



# Load Processed Data

In [12]:
X_train=pd.read_csv('../../data/processed/X_train.csv')
X_test=pd.read_csv('../../data/processed/X_test.csv')
y_train=pd.read_csv('../../data/processed/y_train.csv')
y_test=pd.read_csv('../../data/processed/y_test.csv')

# Feature Groups

| Feature Group      | Features                                                        |
| ------------------ | --------------------------------------------------------------- |
| Frequency Encoding | category, subcategory, assignment_group, assigned_to, opened_by |
| Target Encoding    | caller_id, location, u_symptom, contact_type                    |
| Ordinal Encoding   | impact, urgency, priority                                       |
| Numerical          | Hour                                                            |
| Calendar Features  | Month, Day_of_week                                              |
                                          |


## Preprocessing Pipeline Design

### Traditional ML Pipeline

Missing Value Replacement
        ↓
Ordinal Encoding
        ↓
Frequency Encoding
        ↓
Target Encoding
        ↓
Model

### CatBoost Pipeline

Missing Value Replacement
        ↓
Ordinal Encoding
        ↓
Native Categorical Handling
        ↓
CatBoost

# Custom Transformers

In [13]:

replacement={
    'location': 'Unknown_location',
    'category':'Unknown_category',
    'subcategory':'Unknown_subcategory',
    'u_symptom':'Unknown_symptom',
    'assignment_group':'Unknown_assignment_group',
    'assigned_to':'Unknown_assigned_to',
    'caller_id': 'Unknown_caller_id',
    'opened_by':'Unknown_opened_by'
}

nested_replacement={col: {'?':val} for col, val in replacement.items()}

def replace_missing_value(X):
    X_copy=X.copy()
    return X_copy.replace(nested_replacement)

missing_value_imputer = FunctionTransformer(replace_missing_value)

missing_values_pipeline=Pipeline([('impute_missing_value',missing_value_imputer)])



In [89]:
from sklearn.base import BaseEstimator, TransformerMixin

class BusinessMissingValueTransformer(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.replacement = {
            'location': 'Unknown_location',
            'category': 'Unknown_category',
            'subcategory': 'Unknown_subcategory',
            'u_symptom': 'Unknown_symptom',
            'assignment_group': 'Unknown_assignment_group',
            'assigned_to': 'Unknown_assigned_to',
            'caller_id': 'Unknown_caller_id',
            'opened_by': 'Unknown_opened_by'
        }

        self.missing_tokens = ['?', 'NA', 'N/A', None]

    def fit(self, X, y=None):

        missing_cols = []

        for col in self.replacement.keys():
            if col not in X.columns:
                missing_cols.append(col)

        if missing_cols:
            raise ValueError(
                f"Required missing columns are: {missing_cols}"
            )

        return self

    def transform(self, X):

        X_copy = X.copy()

        for col, replacement_value in self.replacement.items():

            X_copy[col] = X_copy[col].replace(
                self.missing_tokens,
                replacement_value
            )

        return X_copy

In [90]:
missing_transformer = BusinessMissingValueTransformer()

missing_transformer.fit(X_train)

X_train_transformed = missing_transformer.transform(X_train)

In [93]:
(X_train_transformed[
    ['location','category','subcategory',
     'u_symptom','assignment_group',
     'assigned_to','caller_id','opened_by']
].isin(['?','NA','N/A']).sum())

location            0
category            0
subcategory         0
u_symptom           0
assignment_group    0
assigned_to         0
caller_id           0
opened_by           0
dtype: int64

In [87]:
missing_tokens=('?','NA',None,'N/A')
for token in missing_tokens:
    print(token)

?
NA
None
N/A


In [96]:
X_train.shape,X_train_transformed.shape

((113369, 15), (113369, 15))

In [105]:
(X_train_transformed['location']=='Unknown_location')

0         False
1         False
2         False
3         False
4         False
          ...  
113364    False
113365    False
113366    False
113367    False
113368    False
Name: location, Length: 113369, dtype: bool

In [ ]:
X_train_clean=missing_values_pipeline.fit_transform(X_train)

In [ ]:
(X_train_clean=='Unknown_location').sum()
# X_train.shape,X_train_clean.shape

contact_type         0
location            56
category             0
subcategory          0
u_symptom            0
assignment_group     0
assigned_to          0
caller_id            0
opened_by            0
impact               0
urgency              0
priority             0
Hour                 0
Day_of_week          0
Month                0
dtype: int64

## Modeling Strategy

Baseline Models
- Logistic Regression
- Decision Tree
- Random Forest

Boosting Models
- XGBoost
- LightGBM
- CatBoost

Evaluation Metrics
- Precision
- Recall
- F1 Score
- ROC-AUC
- PR-AUC

Cross Validation
- Stratified K-Fold

Hyperparameter Optimization
- Optuna